In [1]:
import json
from openai import OpenAI
from dotenv import load_dotenv
import os
from typing import Dict

load_dotenv("../../.env")

True

### Read pass and fail responses

In [2]:
with open("../results/base-chroma/results.json", "r") as f:
    results = json.load(f)

In [3]:
results

{'pass': {'query': 'How does Jelinek-Mercer smoothing work?',
  'response': {'answer': 'Jelinek-Mercer smoothing is a technique used in language modeling that combines maximum likelihood estimates with a weighted average of lower-order models, specifically using the formula PJM(ngram) = λPML(ngram) + (1 − λ)PJM(n−1gram), where λ is a parameter that balances the contribution of the n-gram and the (n-1)-gram models.',
   'sources': [{'source': '25 Interpolation',
     'usefulness': 'useful',
     'explanation': 'It provides the formula and explanation of how Jelinek-Mercer smoothing combines n-gram and (n-1)-gram probabilities.'},
    {'source': '26 Interpolation: Finding λ',
     'usefulness': 'useful',
     'explanation': 'It discusses the computation of λ values for Jelinek-Mercer smoothing, which is essential for understanding its application.'},
    {'source': '24 Language models n-grams for Language Modeling Handling Unknown Tokens Smoothing n-gram Models',
     'usefulness': 'usef

### Precision and Recall

In [14]:
def precision_k(candidate_list: str, gold_list: list, k: int) -> float:
    deduped_candidate_list = set(candidate_list)
    intersection = set(gold_list).intersection(deduped_candidate_list)
    return len(intersection) / len(deduped_candidate_list)

In [15]:
precision = precision_k(results["pass"]["retrieved_doc_names"], results["pass"]["gold_doc_names"], 10)

In [16]:
def recall_k(candidate_list: str, gold_list: list, k: int) -> float:
    deduped_candidate_list = set(candidate_list)
    intersection = set(gold_list).intersection(deduped_candidate_list)
    return len(intersection) / len(set(gold_list)) if gold_list else 0

In [17]:
recall = recall_k(results["pass"]["retrieved_doc_names"], results["pass"]["gold_doc_names"], 10)

In [18]:
def f1(precision: float, recall: float) -> float:
    return 2 * precision * recall / ( precision + recall ) if precision + recall > 0 else 0

In [19]:
f1(precision, recall)

0.3333333333333333

### Generation - LLM-as-a-Judge

In [10]:
client = OpenAI(
    api_key = os.environ["OPENAI_API_KEY"]
)

In [11]:
def evaluate_answer(client: OpenAI, system_prompt: str, prompt: str, results: Dict) -> Dict:
    prompt = prompt.format(
        question=results["query"],
        retrieved_docs=results["retrieved_texts"],
        generated_answer=results["response"]["answer"]
    )
    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature = 0
    ).choices[0].message.content
    return response

In [12]:
answer_evaluation_system = """
You are a strict but fair evaluator of NLP answers.
"""

answer_evaluation_prompt = """
You are an expert in NLP. Your task is to evaluate a student's answer.

Question:
{question}

Retrieved Context (relevant documents):
{retrieved_docs}

Student's Answer:
{generated_answer}

Evaluation:
1. Is the answer factually correct? (Yes/No)
2. On a scale of 0 to 5, how complete is the answer?
3. Does the answer rely only on the provided context? (Yes/No)
4. Provide an overall accuracy score from 0 to 100.
5. Give a short explanation.

Give me the answer in only a JSON, without ```json fences, like so:
```
{{
    "factual_correctness": yes/no,
    "completeness": score,
    "reliance": yes/no,
    "overall_accruacy": score,
    "explanation": explanation
}}
"""

In [13]:
print(evaluate_answer(client, answer_evaluation_system, answer_evaluation_prompt, results["fail"]))

{
    "factual_correctness": "no",
    "completeness": 0,
    "reliance": "no",
    "overall_accruacy": 0,
    "explanation": "The student's answer states that no useful sources were found, which is incorrect as the retrieved context contains relevant information about mT5 pre-training. The answer does not provide any information or insights related to the question, resulting in a complete lack of completeness and reliance on the provided context."
}
